In [1]:
import os
import sys
from datetime import datetime
from time import strftime
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

import pandas as pd
import seaborn as sn

import hyperspy.api_nogui as hs

import tensorflow as tf

tf.random.set_seed(42)
np.random.seed(42)

In [2]:
from logicalEELS.preprocess import loadDirDM4, findEdgeIndex, alignSpectra, normAUC_inds, backgroundSubtract

In [3]:
# DM4 File Locations C:\Users\holle\Documents\JHU\Data\EELS_Data\\MXene_Inprocess
path_raw = '../../Data/EELS_Data/MXene_Inprocess/MXene_Raw_Rebin_Aligned_All'
path_pca = '../../Data/EELS_Data/MXene_Inprocess/MXene_PCA_Rebin_Aligned_All'

In [ ]:
energy_axis, spectra_raw, filenames_raw = loadDirDM4(path_raw, crop=(390.0, 790.0))

In [ ]:
_, spectra_pca, filenames_pca = loadDirDM4(path_pca, crop=(390.0, 790.0))

In [ ]:
print('Spectra Shapes:')
print('Raw (SIs, pixels, eV bins):', spectra_raw.shape)
print('PCA (SIs, pixels, eV bins):', spectra_pca.shape)
print('Axis (start, step, stop)  :',energy_axis[0], energy_axis[1]-energy_axis[0], energy_axis[-1])

In [7]:
ind_Cr = findEdgeIndex(energy_axis, 567.5, 627.5)
ind_F = findEdgeIndex(energy_axis, 677.0, 737.0)
ind_O = findEdgeIndex(energy_axis, 516.5, 570)

In [8]:
aligned_spectra_raw = alignSpectra(energy_axis, spectra_raw, ind_Cr, 570, 570)
aligned_spectra_pca = alignSpectra(energy_axis, spectra_pca, ind_Cr, 570, 570)

In [9]:
ind_norm = findEdgeIndex(energy_axis, 570.0, 592.5)

aligned_normCr_spectra_raw = normAUC_inds(energy_axis, aligned_spectra_raw, ind_norm)
aligned_normCr_spectra_pca = normAUC_inds(energy_axis, aligned_spectra_pca, ind_norm)

In [ ]:
ind_F_fit = findEdgeIndex(energy_axis, 650, 680)
print('Fit raw data')
backSub_nCr_F_raw = backgroundSubtract(energy_axis, aligned_normCr_spectra_raw, ind_F, ind_F_fit)
print('fit PCA data')
backSub_nCr_F_pca = backgroundSubtract(energy_axis, aligned_normCr_spectra_pca, ind_F, ind_F_fit)

In [11]:
backSub_nCr_F_raw = (50)*backSub_nCr_F_raw.copy()
backSub_nCr_F_pca = (50)*backSub_nCr_F_pca.copy()

X_train = backSub_nCr_F_raw.copy()
X_train = X_train.reshape(-1, np.shape(X_train)[-1])[:,:-1]
Y_train = backSub_nCr_F_pca.copy()
Y_train = Y_train.reshape(-1, np.shape(Y_train)[-1])[:,:-1]

X_train = np.atleast_3d(X_train).astype('float32')
Y_train = np.atleast_3d(Y_train).astype('float32')

In [ ]:
params={
    'INPUT_SHAPE'   :   (240, 1),
    'BATCH_SIZE'    :   16,
    'LATENT_SIZE'   :   16,
    'KERNEL_SIZES'  :   [7, 7, 3, 3],
    'FILTER_SIZES'  :   [16, 32, 32, 64],
    'ALPHA'         :   0.3,
    'DROPOUT'       :   0.2,
    'LR'            :   0.0005,
    'RECON_WEIGHT'  :   1.0,
    'KL_WEIGHT'     :   1.0,
    'CNVRG_WEIGHT'  :   1.0,
}

from logicalEELS.models import createDualVAE
dvae = createDualVAE(params)

dvae.X_encoder.summary()
dvae.Y_encoder.summary()
dvae.decoder.summary()

In [ ]:
denhist = dvae.fit(X_train, Y_train,
                   epochs=30,
                   batch_size=16,
                   verbose=1,
                   shuffle=True,)
plt.title('YB to YB AEC Model Loss')
plt.plot(denhist.history['loss'], label='Train')
plt.plot(denhist.history['reconstruction_loss'], label='Reconstruction')
plt.plot(denhist.history['kl_loss'], label='KL Loss')
plt.plot(denhist.history['convergence_loss'], label='Convergence')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.show()